In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import DeltaTable

## Silver Table Definition

In [0]:
if spark.catalog.tableExists("koine_dev.silver.dim_customer"):
    print("Table exists")
else:
    print(f"Creating a new table: koine_dev.silver.dim_customer")
    spark.sql('''
    CREATE TABLE IF NOT EXISTS koine_dev.silver.dim_customer (
        customer_sk BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY NOT NULL,
        customer_code STRING NOT NULL,
        customer_name STRING,
        city STRING,
        lat DOUBLE,
        lon DOUBLE,
        vpo_class STRING,
        outlet_type STRING,
        source_last_modified TIMESTAMP,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN NOT NULL,
        _silver_updated_at TIMESTAMP NOT NULL,
        hash_diff STRING
    )
    USING delta;'''
    )

## Transformation Logic

In [0]:
def transform(df_bronze, SCD2_COLS):

    ## Dedupe using the last_modified column and ingested_at
    window_spec = (
        Window.partitionBy("customer_code")
        .orderBy(F.col("last_modified").desc(), F.col("_ingested_at").desc())
    )

    df_transformed = (
        df_bronze
        # 1. Clean the business key, dropping nulls AND blank strings
        .withColumn("customer_code", F.trim("customer_code"))
        .filter(F.col("customer_code").isNotNull() & (F.col("customer_code") != ""))
        # 2. Keep only the latest row per customer
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        # 3. Select, clean and cast to match the silver schema
        .select(
            F.col("customer_code"),
            F.trim("customer_name").alias("customer_name"),
            F.initcap(F.trim("city")).alias("city"),          # delhi / DELHI -> Delhi
            F.col("lat").cast("double").alias("lat"),
            F.col("lon").cast("double").alias("lon"),
            F.trim("vpo_class").alias("vpo_class"),
            F.trim("outlet_type").alias("outlet_type"),
            F.col("last_modified").cast("timestamp").alias("source_last_modified"),
        )
        # 4. Hash of the tracked columns, used for change detection in the MERGE
        .withColumn(
            "hash_diff",
            F.sha2(
                F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("~")) for c in SCD2_COLS]),
                256,
            ),
        )
    )

    return df_transformed

## Upsert the data from `bronze` to `silver` schema

In [0]:
SOURCE = "koine_dev.bronze.dim_customer"
TARGET = "koine_dev.silver.dim_customer"

# Columns tracked for SCD2. Any change in these creates a new version.
SCD2_COLS = ["customer_name", "city", "lat", "lon", "vpo_class", "outlet_type"]

source_table = transform(spark.table(SOURCE), SCD2_COLS)

## First check if any records are there in the silver table: 
if spark.table(TARGET).limit(1).count() ==0: 
    print("There is no data in the table, ingesting data")
    df_silver = (
        source_table
        # 5. SCD2 columns for the initial load
        .withColumn("effective_from", F.lit("1900-01-01").cast("timestamp"))
        .withColumn("effective_to", F.lit("9999-12-31").cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("_silver_updated_at", F.current_timestamp())
    )
    df_silver.write.mode('append').saveAsTable(TARGET)
else: 

    ## Loading new deduped records as table
    print("Checking for new records and updates")

    target_table = DeltaTable.forName(spark, TARGET)
    
    ## Mark the old records to is_current_false
    (
        target_table.alias('t')
         .merge(
            source_table.alias('s'),
            's.customer_code = t.customer_code AND t.is_current = true'
        )
        .whenMatchedUpdate(
            condition = 's.hash_diff != t.hash_diff AND s.source_last_modified > t.source_last_modified',
            set = {
                'is_current': 'false',
                'effective_to': 's.source_last_modified',
                '_silver_updated_at': F.current_timestamp()
            }
        )
        .execute()
    )

    ## Find the new records

    current_records = target_table.toDF().filter(F.col('is_current')==True).select('customer_code')
    df_new_records = (
        source_table.join(
            other = current_records, on='customer_code', how='left_anti'
        )
        .withColumn("effective_from", F.col('source_last_modified'))
        .withColumn("effective_to", F.lit("9999-12-31").cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("_silver_updated_at", F.current_timestamp())
    )

    if df_new_records.limit(1).count() > 0:
        print("New records are being ingested...")
        df_new_records.write.mode('append').saveAsTable(TARGET)
    else:
        print("No new upserts found")
    ## Write the new records to the existing table

## New Current Records

In [0]:
spark.table(TARGET).count()

In [0]:
display(spark.table(TARGET).orderBy(F.col("source_last_modified").desc()).limit(10))

## Old Records

In [0]:
display(spark.table(TARGET).filter(F.col('is_current')==False).limit(10))